# Task 4 (Mini Project): High-Imbalance Clinical Disease Predictor Pipeline
**Progree - Machine Learning Internship**

**Objective:** Build an end-to-end classifier that spots early abnormalities in highly skewed medical data, and maximise **recall (sensitivity)** so that few sick patients are missed.

**Dataset:** Breast Cancer Wisconsin (Diagnostic) from the UCI repository (the copy bundled with scikit-learn). The original data is 37% malignant, so I create a **highly imbalanced version (about 11% malignant)** to simulate a screening setting.

**Imbalance handling:** none (reference) vs class weighting vs SMOTE.
**Models:** SVM, Random Forest, XGBoost, each tuned with GridSearchCV.
**Evaluation:** stratified cross-validation score arrays, then a final held-out test set.
**Output:** the last cell writes the 5-6 page report (`.docx`) automatically from the real results.

Run order: Runtime -> Run all. It takes about 3-6 minutes.

## 1. Setup

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [('imbalanced-learn', 'imblearn'), ('xgboost', 'xgboost'), ('python-docx', 'docx')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV, cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (make_scorer, recall_score, precision_score, f1_score, fbeta_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)
from imblearn.pipeline import Pipeline          # applies SMOTE only to training folds
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
STUDENT_NAME = 'Your Name'          # <-- write your name here (used in the report)
TARGET_PREVALENCE = 0.11            # about 11% malignant after imbalancing
os.makedirs('figures', exist_ok=True)

## 2. Load the dataset and create the imbalance
In the original data 212 of 569 samples are malignant (37%). To get a highly skewed, screening-like dataset I keep all 357 benign samples and a random subset of malignant ones. **This imbalance is simulated; the features and labels are real UCI data.**
Label convention here: **1 = malignant (the positive class we must not miss), 0 = benign.**

In [ ]:
raw = load_breast_cancer(as_frame=True)
X_full = raw.data
y_full = (raw.target == 0).astype(int)        # scikit-learn codes malignant as 0, so flip it

rng = np.random.RandomState(SEED)
benign_idx = np.where(y_full == 0)[0]
malig_idx = np.where(y_full == 1)[0]
n_mal = int(round(TARGET_PREVALENCE / (1 - TARGET_PREVALENCE) * len(benign_idx)))
keep = np.sort(np.concatenate([benign_idx, rng.choice(malig_idx, n_mal, replace=False)]))

X = X_full.iloc[keep].reset_index(drop=True)
y = y_full.iloc[keep].reset_index(drop=True)

print('Original  : benign', int((y_full == 0).sum()), '| malignant', int((y_full == 1).sum()),
      f'({y_full.mean():.1%} malignant)')
print('Imbalanced: benign', int((y == 0).sum()), '| malignant', int((y == 1).sum()),
      f'({y.mean():.1%} malignant)')
print('Features  :', X.shape[1])
display(X.iloc[:, :6].describe().T)
print('Missing values:', int(X.isna().sum().sum()))

fig, ax = plt.subplots(figsize=(5.5, 3.4))
vals = [(y_full == 0).sum(), (y_full == 1).sum(), (y == 0).sum(), (y == 1).sum()]
pos = [0, 1, 2.6, 3.6]
ax.bar(pos, vals, color=['#7a8a99', '#d62728', '#7a8a99', '#d62728'])
ax.set_xticks(pos); ax.set_xticklabels(['benign', 'malignant', 'benign', 'malignant'])
ax.text(0.5, max(vals) * 1.17, 'Original', ha='center', fontweight='bold'); ax.text(3.1, max(vals) * 1.17, 'Imbalanced (used)', ha='center', fontweight='bold')
for p, v in zip(pos, vals):
    ax.text(p, v + 5, str(v), ha='center', fontsize=9)
ax.set_ylim(0, max(vals) * 1.3); ax.set_ylabel('samples'); ax.set_title('Class distribution')
plt.tight_layout(); plt.savefig('figures/fig1_class_distribution.png', dpi=170); plt.show()

## 3. Train / test split
A stratified 75/25 split. The test set is **not touched** until the very end: all tuning and model selection use cross-validation on the training part only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
print('Train:', X_train.shape, '| malignant =', int(y_train.sum()), f'({y_train.mean():.1%})')
print('Test :', X_test.shape, '| malignant =', int(y_test.sum()), f'({y_test.mean():.1%})')

## 4. Pipelines, imbalance strategies and hyperparameter grids
Three strategies are compared for every model:
- **none**: no imbalance handling (reference)
- **class_weight**: errors on the rare class cost more (`balanced` for SVM, `balanced_subsample` for Random Forest, `scale_pos_weight = negatives/positives` for XGBoost)
- **smote**: SMOTE creates synthetic malignant samples. It sits **inside the pipeline**, so inside cross-validation it only sees training folds (no data leakage).

**Metric used for tuning:** recall alone is trivially maximised by labelling everyone malignant, so models are tuned for the **F2 score** (recall counts twice as much as precision). Recall and precision are always reported separately as well.

In [ ]:
neg_pos_ratio = float((y_train == 0).sum() / (y_train == 1).sum())

def build(model_name, strategy):
    steps = [('scaler', StandardScaler())]
    if strategy == 'smote':
        steps.append(('smote', SMOTE(random_state=SEED)))
    cw = strategy == 'class_weight'
    if model_name == 'SVM':
        clf = SVC(kernel='rbf', class_weight='balanced' if cw else None, random_state=SEED)
    elif model_name == 'Random Forest':
        clf = RandomForestClassifier(class_weight='balanced_subsample' if cw else None,
                                     random_state=SEED, n_jobs=1)
    else:
        clf = XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=1,
                            scale_pos_weight=neg_pos_ratio if cw else 1.0)
    steps.append(('clf', clf))
    return Pipeline(steps)

grids = {
    'SVM': {'clf__C': [0.1, 1, 10, 100], 'clf__gamma': ['scale', 0.01, 0.001]},
    'Random Forest': {'clf__n_estimators': [200, 400], 'clf__max_depth': [None, 5, 10],
                      'clf__min_samples_leaf': [1, 3]},
    'XGBoost': {'clf__n_estimators': [100, 300], 'clf__max_depth': [2, 3, 5],
                'clf__learning_rate': [0.05, 0.1, 0.3]},
}
MODELS = list(grids)
STRATEGIES = ['none', 'class_weight', 'smote']

scoring = {
    'recall': 'recall',
    'precision': make_scorer(precision_score, zero_division=0),
    'f2': make_scorer(fbeta_score, beta=2),
}

## 5. Hyperparameter tuning (GridSearchCV, stratified 5-fold)
Every model x strategy pair is tuned separately (9 searches).

In [ ]:
tuned = {}
cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for m in MODELS:
    for s in STRATEGIES:
        gs = GridSearchCV(build(m, s), grids[m], scoring=scoring, refit='f2',
                          cv=cv_tune, n_jobs=-1)
        gs.fit(X_train, y_train)
        tuned[(m, s)] = gs
        i = gs.best_index_
        print(f"{m:14s} {s:13s} CV recall {gs.cv_results_['mean_test_recall'][i]:.3f} | "
              f"CV precision {gs.cv_results_['mean_test_precision'][i]:.3f} | "
              f"best params {gs.best_params_}")

## 6. Cross-validation score arrays
Each tuned pipeline is evaluated with **repeated stratified 5-fold cross-validation (3 repeats = 15 scores per metric)**. The arrays below are the cross-validation arrays; the table shows their mean and standard deviation.
Note: the hyperparameters were chosen on the same training data, so these CV numbers are slightly optimistic. The held-out test set in section 8 is the unbiased check.

In [ ]:
cv_eval = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_arrays, rows = {}, []
for (m, s), gs in tuned.items():
    res = cross_validate(gs.best_estimator_, X_train, y_train, cv=cv_eval, scoring=scoring, n_jobs=-1)
    cv_arrays[(m, s)] = {k: res['test_' + k] for k in scoring}
    a = cv_arrays[(m, s)]
    rows.append({'Model': m, 'Strategy': s,
                 'CV Recall': a['recall'].mean(), 'Recall SD': a['recall'].std(),
                 'CV Precision': a['precision'].mean(), 'CV F2': a['f2'].mean()})
results = pd.DataFrame(rows)
display(results.round(3))

# the raw arrays for one configuration, as an example
ex = ('XGBoost', 'smote')
print('Example: 15 recall scores for', ex, '->', np.round(cv_arrays[ex]['recall'], 2))

In [ ]:
long = pd.DataFrame([{'Model': m, 'Strategy': s, 'Recall': v}
                     for (m, s), a in cv_arrays.items() for v in a['recall']])
plt.figure(figsize=(8.2, 4.2))
sns.boxplot(data=long, x='Model', y='Recall', hue='Strategy', palette=['#9aa5b1', '#2a7de1', '#e08a1e'])
plt.title('Cross-validated recall (15 folds per box)'); plt.ylim(0, 1.05)
plt.legend(title='Imbalance strategy', loc='lower right', fontsize=8)
plt.tight_layout(); plt.savefig('figures/fig2_cv_recall.png', dpi=170); plt.show()

## 7. Choose the final configuration per model
For each model the imbalance strategy with the best mean cross-validated F2 is selected (ties go to higher recall). The test set plays no role here.

In [ ]:
chosen = {}
for m in MODELS:
    sub = results[results.Model == m].sort_values(['CV F2', 'CV Recall'], ascending=False)
    chosen[m] = sub.iloc[0]['Strategy']
print('Chosen strategy per model:', chosen)

## 8. Final evaluation on the held-out test set

In [ ]:
def get_score(est, Xs):
    return est.predict_proba(Xs)[:, 1] if hasattr(est, 'predict_proba') else est.decision_function(Xs)

test_rows, test_scores, test_cms = [], {}, {}
for m in MODELS:
    est = tuned[(m, chosen[m])].best_estimator_
    pred = est.predict(X_test)
    score = get_score(est, X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    test_scores[m] = score; test_cms[m] = confusion_matrix(y_test, pred)
    test_rows.append({'Model': m, 'Strategy': chosen[m],
                      'Recall': recall_score(y_test, pred), 'Precision': precision_score(y_test, pred, zero_division=0),
                      'F1': f1_score(y_test, pred), 'ROC-AUC': roc_auc_score(y_test, score),
                      'PR-AUC': average_precision_score(y_test, score),
                      'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn})
test_df = pd.DataFrame(test_rows)
display(test_df.round(3))

fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.3))
for a, m in zip(ax, MODELS):
    sns.heatmap(test_cms[m], annot=True, fmt='d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['benign', 'malignant'], yticklabels=['benign', 'malignant'])
    a.set_title(f'{m} ({chosen[m]})', fontsize=9); a.set_xlabel('Predicted'); a.set_ylabel('True')
plt.tight_layout(); plt.savefig('figures/fig3_confusion_matrices.png', dpi=170); plt.show()

plt.figure(figsize=(5.6, 4.2))
for m in MODELS:
    p, r, _ = precision_recall_curve(y_test, test_scores[m])
    plt.plot(r, p, label=f"{m} (AP={average_precision_score(y_test, test_scores[m]):.2f})")
plt.axhline(y_test.mean(), color='grey', ls=':', label='no-skill baseline')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-recall curves (test set)')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig('figures/fig4_pr_curves.png', dpi=170); plt.show()

## 9. Generate the 5-6 page report (.docx)
This cell writes the whole report from the results above, so the text and numbers always match what you actually ran.

In [ ]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

n_pos_test = int(y_test.sum()); n_test = len(y_test)
n_pos_train = int(y_train.sum()); n_train = len(y_train)
best_row = results.assign(_k=list(zip(results['CV F2'], results['CV Recall']))).sort_values('_k').iloc[-1]
best_m, best_s = best_row['Model'], best_row['Strategy']
best_test = test_df[test_df.Model == best_m].iloc[0]
strat_avg = results.groupby('Strategy')['CV Recall'].mean()
model_avg = results.groupby('Model')['CV Recall'].mean()

doc = Document()
sec = doc.sections[0]
sec.left_margin = sec.right_margin = Inches(1); sec.top_margin = sec.bottom_margin = Inches(0.9)
st = doc.styles['Normal']; st.font.name = 'Calibri'; st.font.size = Pt(11)
st.paragraph_format.space_after = Pt(6); st.paragraph_format.line_spacing = 1.1

def H(t, lvl=1):
    h = doc.add_heading(t, level=lvl)
    for r in h.runs:
        r.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F); r.font.name = 'Calibri'
def P(t, italic=False, align=None):
    p = doc.add_paragraph(); r = p.add_run(t); r.italic = italic
    if align == 'center': p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    return p
def FIG(path, cap, width):
    doc.add_picture(path, width=Inches(width))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    c = P(cap, italic=True, align='center'); c.runs[0].font.size = Pt(9.5)
def TABLE(df, cap, widths=None):
    c = P(cap, italic=True); c.runs[0].font.size = Pt(9.5); c.paragraph_format.keep_with_next = True
    t = doc.add_table(rows=1, cols=len(df.columns)); t.style = 'Light Grid Accent 1'
    for i, col in enumerate(df.columns):
        t.rows[0].cells[i].text = str(col)
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = f'{v:.3f}' if isinstance(v, (float, np.floating)) else str(v)
    for row in t.rows:
        for cell in row.cells:
            for p in cell.paragraphs:
                p.paragraph_format.space_after = Pt(0)
                for r in p.runs: r.font.size = Pt(9)
    doc.add_paragraph().paragraph_format.space_after = Pt(2)

# ---- title block
t = doc.add_paragraph(); t.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = t.add_run('High-Imbalance Clinical Disease Predictor Pipeline'); r.bold = True; r.font.size = Pt(20); r.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)
P('Mini Project - Machine Learning Internship at Progree', align='center')
P(f'Prepared by: {STUDENT_NAME}   |   Date: {datetime.date.today():%d %B %Y}', align='center')

H('Abstract')
P(f"Early detection of disease depends on classifiers that rarely miss a sick patient, yet medical data is usually heavily skewed towards healthy cases. "
  f"This project builds an end-to-end pipeline on the UCI Breast Cancer Wisconsin (Diagnostic) data, which was subsampled to {y.mean():.1%} malignant cases to simulate a screening population. "
  f"Three imbalance strategies (none, class weighting, SMOTE) were combined with three models (SVM, Random Forest, XGBoost), each tuned by grid search and evaluated with repeated stratified 5-fold cross-validation, "
  f"with recall (sensitivity) as the main concern. The configuration with the best cross-validated F2 score was {best_m} with {best_s.replace('_', ' ')} "
  f"(CV recall {best_row['CV Recall']:.3f} +/- {best_row['Recall SD']:.3f}). On the untouched test set it detected {int(best_test['TP'])} of {n_pos_test} malignant cases "
  f"(recall {best_test['Recall']:.3f}) with precision {best_test['Precision']:.3f}.")

H('1. Introduction')
P("In screening problems the disease is rare, so a dataset contains many more healthy than sick examples. This breaks the usual way of judging a model. "
  f"With {y.mean():.0%} positives, a classifier that labels every patient as healthy is {1 - y.mean():.0%} accurate but finds no cancer at all. Accuracy is therefore misleading, and the cost of errors is also unequal: "
  "a missed malignant case (false negative) can delay treatment, while a false alarm (false positive) usually leads to a follow-up test. For this reason the project concentrates on recall, also called sensitivity, "
  "which is the fraction of truly malignant cases that the model finds.")
P("The objectives were: (1) build a reproducible classification pipeline for a highly imbalanced diagnostic dataset; (2) compare class weighting and SMOTE against no imbalance handling; "
  "(3) tune and compare SVM, Random Forest and XGBoost; and (4) report recall and related metrics using cross-validation score arrays and a final held-out test set.")

H('2. Dataset')
P("The Breast Cancer Wisconsin (Diagnostic) dataset from the UCI Machine Learning Repository contains 569 samples. Each sample describes cell nuclei in a digitised image of a fine-needle aspirate of a breast mass. "
  "Ten characteristics (such as radius, texture, smoothness and concavity) are summarised by their mean, standard error and worst value, giving 30 numeric features with no missing values. "
  "The label is malignant or benign. In the original data 212 samples (37%) are malignant, which is only moderately imbalanced.")
P(f"To create a highly imbalanced screening scenario, all 357 benign samples were kept and {int(y.sum())} malignant samples were chosen at random, giving {len(y)} samples with {y.mean():.1%} malignant. "
  "The features and labels are real; only the class ratio was changed, and this should be kept in mind when reading the results. "
  f"The data was split with stratification into a training set ({n_train} samples, {n_pos_train} malignant) and a test set ({n_test} samples, {n_pos_test} malignant). "
  "The test set was used only once, at the end.")
ds = pd.DataFrame({'Item': ['Source', 'Features', 'Samples used', 'Malignant (positive class)', 'Benign', 'Training set', 'Test set'],
                   'Value': ['UCI Breast Cancer Wisconsin (Diagnostic)', '30 numeric, no missing values', str(len(y)),
                             f'{int(y.sum())} ({y.mean():.1%})', str(int((y == 0).sum())),
                             f'{n_train} ({n_pos_train} malignant)', f'{n_test} ({n_pos_test} malignant)']})
TABLE(ds, 'Table 1. Dataset summary.')
FIG('figures/fig1_class_distribution.png', 'Figure 1. Class distribution before and after creating the imbalance.', 4.3)

H('3. Methodology')
H('3.1 Pipeline and data leakage', 2)
P("Every model is a single pipeline: standardisation (zero mean, unit variance), optional SMOTE, then the classifier. The pipeline is refitted inside each cross-validation fold, "
  "so scaling statistics and synthetic samples are created from the training folds only. Applying SMOTE or scaling before the split would leak information from the validation fold and inflate the scores.")
H('3.2 Imbalance strategies', 2)
P("None: the data is used as it is, as a reference. Class weighting: errors on the malignant class are penalised more (class_weight='balanced' for SVM, 'balanced_subsample' for Random Forest, and scale_pos_weight equal to the ratio of benign to malignant training samples for XGBoost). "
  "SMOTE (Synthetic Minority Over-sampling Technique): new malignant samples are created by interpolating between a real malignant sample and one of its nearest malignant neighbours, until both classes have the same size in the training fold.")
H('3.3 Models and tuning', 2)
P("An RBF-kernel Support Vector Machine (grid over C and gamma), a Random Forest (number of trees, maximum depth, minimum leaf size) and an XGBoost gradient-boosted tree model (number of trees, depth, learning rate) were used. "
  "Each of the nine model and strategy combinations was tuned with GridSearchCV using stratified 5-fold cross-validation on the training set.")
H('3.4 Evaluation metrics', 2)
P("Recall = TP / (TP + FN) measures how many malignant cases are found. Precision = TP / (TP + FP) measures how many alarms are correct. "
  "Maximising recall alone is trivial, because labelling every patient as malignant gives a recall of 1.0, so tuning used the F2 score, which weights recall twice as much as precision. "
  "Recall and precision are reported separately in all tables. Final cross-validation scores come from repeated stratified 5-fold cross-validation with 3 repeats, which gives 15 scores per metric; the tables show their mean and standard deviation. "
  "ROC-AUC and the area under the precision-recall curve (PR-AUC) are reported on the test set, where PR-AUC is the more informative of the two for rare positives.")

H('4. Results')
H('4.1 Cross-validation results', 2)
cvt = results.rename(columns={'CV Recall': 'Recall (mean)', 'Recall SD': 'Recall (SD)', 'CV Precision': 'Precision (mean)', 'CV F2': 'F2 (mean)'})
TABLE(cvt, 'Table 2. Repeated stratified 5-fold cross-validation (15 scores per cell) on the training set.')
P(f"Averaged over the three models, the mean cross-validated recall was {strat_avg['none']:.3f} with no imbalance handling, {strat_avg['class_weight']:.3f} with class weighting and {strat_avg['smote']:.3f} with SMOTE. "
  f"Averaged over the three strategies, the mean recall was {model_avg['SVM']:.3f} for SVM, {model_avg['Random Forest']:.3f} for Random Forest and {model_avg['XGBoost']:.3f} for XGBoost. "
  f"The best cross-validated F2 score belonged to {best_m} with {best_s.replace('_', ' ')} (F2 {best_row['CV F2']:.3f}, recall {best_row['CV Recall']:.3f}, precision {best_row['CV Precision']:.3f}).")
FIG('figures/fig2_cv_recall.png', 'Figure 2. Distribution of the 15 cross-validated recall scores for each model and imbalance strategy.', 5.6)

H('4.2 Held-out test results', 2)
tt = test_df[['Model', 'Strategy', 'Recall', 'Precision', 'F1', 'ROC-AUC', 'PR-AUC', 'TP', 'FN', 'FP']]
TABLE(tt, f'Table 3. Test-set performance of the selected configuration of each model ({n_pos_test} malignant and {n_test - n_pos_test} benign samples).')
FIG('figures/fig3_confusion_matrices.png', 'Figure 3. Confusion matrices on the test set.', 6.3)
P(f"On the test set, {best_m} found {int(best_test['TP'])} of the {n_pos_test} malignant cases and missed {int(best_test['FN'])}, with {int(best_test['FP'])} false alarms among {n_test - n_pos_test} benign samples. "
  f"Its ROC-AUC was {best_test['ROC-AUC']:.3f} and its PR-AUC {best_test['PR-AUC']:.3f}.")
FIG('figures/fig4_pr_curves.png', 'Figure 4. Precision-recall curves on the test set. The dotted line is the no-skill baseline, equal to the share of malignant cases.', 4.3)

H('5. Discussion')
P("The cross-validation arrays show both the average level and the spread of recall. A model whose boxes in Figure 2 are high and narrow finds malignant cases consistently, whereas a wide box means the result depends strongly on which patients fall in the training folds. "
  "With only a few dozen malignant cases in total, the spread is expected to be noticeable. Comparing the strategies on the same model shows how much the rare class benefits from extra attention; "
  "the averages in Section 4.1 summarise this, and the differences should be read together with the standard deviations before concluding that one strategy is better.")
P(f"The test set contains only {n_pos_test} malignant samples, so a single additional missed case changes recall by about {100 / n_pos_test:.0f} percentage points. "
  "The test numbers are therefore a plausibility check on the cross-validation results and not a precise estimate of future performance. "
  "The trade-off between recall and precision is visible in the precision-recall curves: raising recall by lowering the decision threshold will usually add false alarms, and the acceptable balance is a clinical decision.")

H('6. Limitations')
P("(1) The class imbalance was created artificially from a dataset that is only moderately imbalanced; real screening data has additional difficulties such as noisy labels and shifting populations. "
  "(2) The dataset is small, so scores have wide uncertainty. (3) Hyperparameters were selected on the same folds that give the cross-validation scores, which makes those scores slightly optimistic; nested cross-validation would remove this effect. "
  "(4) All models use the default decision threshold of 0.5; tuning the threshold for a target sensitivity was not done. (5) The study uses one dataset and has not been validated clinically, so it must not be used for real diagnosis.")

H('7. Conclusion')
P(f"An end-to-end, leakage-free pipeline was built that handles class imbalance, tunes SVM, Random Forest and XGBoost, and evaluates them with repeated cross-validation and a held-out test set. "
  f"The best configuration by cross-validated F2 score was {best_m} with {best_s.replace('_', ' ')}, reaching a test recall of {best_test['Recall']:.3f} at a precision of {best_test['Precision']:.3f}. "
  "Future work includes nested cross-validation, threshold tuning for a fixed sensitivity target, calibration of predicted probabilities, and validation on a genuinely imbalanced clinical dataset such as the Pima Indians diabetes records.")

H('References')
for ref in [
    "Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1995). Breast Cancer Wisconsin (Diagnostic) Data Set. UCI Machine Learning Repository.",
    "Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic Minority Over-sampling Technique. Journal of Artificial Intelligence Research, 16, 321-357.",
    "Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. Proceedings of KDD 2016, 785-794.",
    "Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, 12, 2825-2830.",
    "Lemaitre, G., Nogueira, F., & Aridas, C. K. (2017). Imbalanced-learn: A Python Toolbox to Tackle the Curse of Imbalanced Datasets in Machine Learning. Journal of Machine Learning Research, 18(17), 1-5.",
]:
    p = P(ref); p.runs[0].font.size = Pt(10)

REPORT = 'Task4_Clinical_Disease_Predictor_Report.docx'
doc.save(REPORT)
results.to_csv('t4_cv_results.csv', index=False); test_df.to_csv('t4_test_results.csv', index=False)
print('Report saved:', REPORT)

In [ ]:
try:
    from google.colab import files
    files.download(REPORT)
except ImportError:
    print('Not running in Colab; the report is in the current folder.')